In [1]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from env.charging_env import ChargingStationEnv
from metrics.validate import report_queueing_laws, validate_episode

from policy.power.proportional import ProportionalPower
from policy.power.static import StaticPower
from policy.queue.fifo import FIFOQueuePolicy
from policy.queue.lowest_soc_diff import LowestSoCDiffQueuePolicy
from visualization.ev_curves import plot_ev_theory_vs_sim
from visualization.pile_power import plot_pile_connector_power
from config import MAX_TIME
from simulation.arrivals import generate_arrivals

from __future__ import annotations

from offline_cl_opt import (
    StationSpec,
    build_cl_model,
    solve_cl_model,
    vehicles_from_evs,
)
from offline_cl_opt.solution import extract_solution


In [ ]:
# Shared station / traffic inputs (simulation and offline use the same layout).
SCENARIO = {
    "n_piles": 2, # Typical 2 - 4
    "n_connectors": 2, # Typical 2
    "n_modules": 6, # 4 - 8
    "p_module": 25.0, # 25 - 50 kW; pile power 150 - 400 kW
    "queue_capacity": 1000, #
}

# Simulation controls
HR2MIN = 60
MAX_TIME = HR2MIN * 2   # length of the MEASURED phase (minutes) -- see WARMUP_PERIOD below
MEAN_INTERARRIVAL = 15.0
DELTA = 2.0
# None: keep continuous exponential times.
# 1: nearest minute; 2/3/5: nearest multiple of that many minutes.
DELTA_ARR = DELTA
BATTERY_CAP_OPTIONS = [75] # Typical 75, test [50, 75, 100]
BATTERY_CAP_OPTIONS = [cap * HR2MIN for cap in BATTERY_CAP_OPTIONS]
SEED = 39
POLICY_SEED = 12
QUEUE_POLICY = FIFOQueuePolicy()
POWER_POLICY = ProportionalPower()

# --- Warm-up period -- usage example ------------------------------------
# See simulation/engine.py's own module docstring ("Warm-up period") and
# metrics/metrics_tracker.py's for the full mechanics. Short version: the
# DES runs WARMUP_PERIOD minutes first, then MAX_TIME more (the "measured"
# phase); queue/pile state at the boundary carries straight over (nobody is
# reset), so the measured phase doesn't start from an empty station -- lets
# transient start-up effects (empty queue, all piles idle) settle out
# before the window you actually report metrics over.
#
# ChargingStationEnv(..., warmup_period=None) -- the default -- reads
# config.WARMUP_PERIOD (0.0 out of the box, i.e. no warm-up at all,
# fully backward compatible). Pass an explicit value, as below, to
# override it for this run without touching config.py.
WARMUP_PERIOD = MAX_TIME * 20  # here: a long warm-up relative to MAX_TIME, deliberately, to reach steady state

# If True, empty the live QUEUE only (never vehicles already charging --
# see metrics.in_service_at_warmup_end below) at the exact instant warm-up
# ends, so the measured window's own queueing dynamics don't inherit
# whatever backlog built up during warm-up. Off by default: usually you
# want the measured phase to see a realistic, already-loaded system, not
# an artificially emptied queue.
FLUSH_QUEUE_AT_WARMUP = False

# 1. Generate the EV list yourself (same object format the simulation uses internally)
# max_time here must cover the WHOLE run (warm-up + measured) -- oversample
# generously since this is a one-shot draw, not resampled per phase.
rng_arrival = np.random.default_rng(SEED)
evs = generate_arrivals(
    mean_interarrival=MEAN_INTERARRIVAL,
    max_time=(WARMUP_PERIOD + MAX_TIME) * 5, rng=rng_arrival,
    battery_cap_options=BATTERY_CAP_OPTIONS,
    delta_arr=DELTA_ARR,
    )


In [ ]:
#Feed it to the simulation: mean_interarrival=None so the external list is used
env = ChargingStationEnv(
    **SCENARIO,
    power_policy=POWER_POLICY,
    mean_interarrival=None,   # None -> use `arrivals`, not internal sampling
    arrivals=evs,
    max_time=MAX_TIME,
    battery_cap_options=BATTERY_CAP_OPTIONS,
    delta_arr=DELTA_ARR,
    warmup_period=WARMUP_PERIOD,              # usage example -- see the cell above
    flush_queue_at_warmup=FLUSH_QUEUE_AT_WARMUP,
)
obs, info = env.reset(seed=SEED)

done = False
rng = np.random.default_rng(POLICY_SEED)
policy = QUEUE_POLICY
total_reward = 0.0

t0 = time.perf_counter()
while not done:
    mask = env.action_masks()
    ev, pile_id = policy.decide(obs, mask, rng, env.engine.station)
    obs, reward, done, _, info = env.step(pile_id, ev=ev)
    total_reward += reward
sim_runtime = time.perf_counter() - t0

m = env.engine.metrics
finished = m.finished_evs
arrived = m.arrived_evs
dropped = m.dropped_evs

# Kept as whole-run scalars (downstream cells reference these by name) --
# same values as the "whole-run" side of the two-window print below.
sim_total_sojourn = float(sum(ev.departure_time - ev.arrival_time for ev in finished))
sim_mean_sojourn = m.mean_sojourn()

print(f"Finished EVs: {len(finished)}")
print(f"Arrived EVs:  {len(arrived)}")
print(f"Dropped EVs:  {len(dropped)}")
print(f"Total reward: {total_reward:.3f}")
print(f"Sim time (episode clock, TOTAL incl. warm-up): {env.engine.current_time:.1f} min "
      f"(warm-up={env.engine.warmup_period:.1f} + measured={env.engine.measurement_horizon:.1f})")
print(f"Sim wall-clock runtime:   {sim_runtime:.3f} s")

# --- Metrics, two windows -------------------------------------------------
# Everything below is reported BOTH ways: over the whole simulated run
# (including warm-up) and restricted to the measured phase only
# (post_warmup_only=True / since=m.warmup_period). See MetricsTracker's own
# module docstring, "Warm-up period", for exactly which quantities are
# time-windowed (L, Q, utilization, energy) vs. cohort-filtered (per-EV
# wait/service/sojourn -- excludes anyone who *arrived* during warm-up,
# even if they finish after it) by `since`.
print("\n=== Sojourn: whole run vs. measured-phase-only ===")
print(f"  Sim total sojourn  whole-run={sim_total_sojourn:.2f} min   "
      f"measured-only={m.finished_time_arrays(m.warmup_period)[2].sum():.2f} min")
print(f"  Sim mean  sojourn  whole-run={sim_mean_sojourn:.2f} min   "
      f"measured-only={m.mean_sojourn(m.warmup_period):.2f} min")
print(f"  Avg L  whole-run={m.average_L():.3f}   measured-only={m.average_L(m.warmup_period):.3f}")
print(f"  Avg Q  whole-run={m.average_Q():.3f}   measured-only={m.average_Q(m.warmup_period):.3f}")

# validate_episode's conservation check accounts for the warm-up flush on
# its own (arrived = finished + dropped + flushed + queued + plugged); it
# is always a whole-run check (see its own docstring -- it's a live-state
# audit, not something that has a "since" version).
rep_val = validate_episode(env, verbose=True)

# report_queueing_laws(post_warmup_only=...) is the single switch for the
# Little's-law summary: False (default) = whole run; True = measured phase
# only. Both calls below share the exact same MetricsTracker.queueing_summary
# under the hood -- see metrics/validate.py.
print("\n--- Whole run ---")
report_full = report_queueing_laws(env, verbose=True, post_warmup_only=False)
print("\n--- Measured phase only (post-warm-up) ---")
report_post = report_queueing_laws(env, verbose=True, post_warmup_only=True)

# --- Boundary-condition lists (for a future optimization run confined to
# just the measured window -- see MetricsTracker's own module docstring) --
print(f"\nqueued_at_warmup_end:     {len(m.queued_at_warmup_end)} vehicle(s) waiting when the measured phase began")
print(f"in_service_at_warmup_end: {len(m.in_service_at_warmup_end)} vehicle(s) already mid-charge when the measured phase began")
print(f"arrived_post_warmup:      {len(m.arrived_post_warmup)} vehicle(s) that arrived during the measured phase")
print(f"flushed_evs:              {len(m.flushed_evs)} vehicle(s) discarded from the queue at the boundary "
      f"(only nonempty if FLUSH_QUEUE_AT_WARMUP=True)")



=== Episode validation ===
  arrived=203, finished=201, dropped=0, flushed=0, queued=0, plugged=2
  [PASS] Finished: s_current ~= s_f: ok for all finished EVs
  [PASS] Finished: energy_received ~= c_b*(s_f-s_i)/HR2MIN: ok for all finished EVs
  [PASS] Finished: energy_received ~= energy_received_2: ok for all finished EVs
  [PASS] Finished: departure_time >= service_start_time >= arrival_time: ok for all finished EVs
  [PASS] Finished: pile_tracker / connector_id_tracker set: ok for all finished EVs
  [PASS] Finished: p_act <= p_req in charge_trace: ok for all finished EVs
  [PASS] Dropped: never got service_start_time / empty charge_trace: ok for all dropped EVs
  [PASS] Flushed: never got service_start_time / empty charge_trace: ok for all flushed EVs
  [PASS] arrived = finished + dropped + flushed + queued + plugged: 203 vs 201+0+0+0+2=203

=== Queueing laws (post-warm-up, t>=2400) ===
  T=240.0 min, finished=14, c=4 servers
  lambda_eff=0.058333 /min
  E[W]=27.9124 min, E[W_q]=0.3

In [ ]:
# Offline IP + continuous relaxations on the same station layout and the same
# arrival stream the episode already realized (no synthetic arrivals after
# simulation end). VehicleData uses each EV's arrival, battery, s_i, s_f.
cl_station = StationSpec(
    n_piles=SCENARIO["n_piles"],
    n_connectors=SCENARIO["n_connectors"],
    n_modules=SCENARIO["n_modules"],
    p_module=SCENARIO["p_module"],
)

# Offline MILP / RP controls (same arrivals + station as the finished episode)

MIP_GAP = 1e-3
TIME_LIMIT = 600.0  # seconds per solve; raise if the gap does not close
# Manual completion horizon for the offline models (minutes). This models
# just the MEASURED phase -- OFFLINE_HORIZON = MAX_TIME (not the total
# warm-up + measured duration the DES actually ran for; see engine.max_time
# below). Must be long enough that every arrived EV can finish.
OFFLINE_HORIZON = MAX_TIME
# The DES always runs to its own total horizon (warm-up + measured) exactly
# -- this checks that actually happened (a genuine invariant regardless of
# warm-up), not that it matches OFFLINE_HORIZON (which by design only
# covers the measured phase, not warm-up).
if not np.isclose(env.engine.max_time, float(env.engine.current_time)):
    raise ValueError("Engine's total horizon and actual finish time are not equal")

# Vehicles for the offline model: only arrived_post_warmup (not the whole
# run's arrived_evs, and not the raw oversampled `evs` draw) -- since
# OFFLINE_HORIZON only spans the MEASURED phase, feeding it a warm-up-era
# arrival would be wrong twice over: it wasn't part of the window being
# modeled, and (see the arrival_time shift below) its clock value predates
# what "t=0" means for this offline instance anyway.
#
# arrival_time on every EV is still stamped in *whole-run* clock (e.g. 2450
# if warmup_period=2400), but the offline model's own horizon starts at
# t=0 for the measured window -- shift every vehicle's `a` back by
# warmup_period so k_j = ceil(a_j/delta) lands inside [0, OFFLINE_HORIZON)
# instead of raising "release slot falls outside the horizon".
#
# NOTE this deliberately does NOT include queued_at_warmup_end /
# in_service_at_warmup_end (vehicles already waiting/charging when the
# measured phase began) as boundary-condition vehicles -- per instructions,
# the optimization model itself isn't being changed yet to consume those.
# This offline instance is therefore only a partial picture of the measured
# window whenever WARMUP_PERIOD > 0 and the station wasn't empty at the
# boundary -- fine for exercising the pipeline, not yet a fair comparison
# against the simulation's own (boundary-inclusive) measured-phase stats.
import dataclasses
vehicles = [
    dataclasses.replace(v, a=v.a - env.engine.warmup_period)
    for v in vehicles_from_evs(env.engine.metrics.arrived_post_warmup)
]
print(
    f"Offline instance: J={len(vehicles)} vehicles "
    f"(arrived during the measured phase only, warmup_period={env.engine.warmup_period:.1f} min), "
    f"horizon T={OFFLINE_HORIZON:.1f} min, delta={DELTA:g} min, "
)

In [34]:

from offline_cl_dw import solve_by_decomposition
# mean-sojourn uncertainty (minutes) = δ · g / J
# gap = mean-soj uncertainty * J / δ
# To target an m-minute mean-sojourn tolerance:
# gap_tolerance = m * n_vehicles / delta
# n_vehicles = MAX_TIME / MEAN_INTERARRIVAL
# gap_tolerance = m * MAX_TIME / (MEAN_INTERARRIVAL * delta)
dw_solution, dw_colgen = solve_by_decomposition(
    vehicles, cl_station, delta=DELTA, horizon_minutes=OFFLINE_HORIZON,
    progress=True,
    gamma=0.0,
    time_limit=None,
    purge_every=None,
    gap_tolerance=.000001,
)

print(f"\nbracket: [{dw_solution.LB:.3f}, {dw_solution.UB:.3f}]  "
      f"gap={dw_solution.gap:.3f}  converged={dw_solution.converged}  "
      f"iterations={dw_solution.iterations}")
# rmp_gap is the colgen stopping-criterion gap (z_RMP - LB) -- a diagnostic
# of how close column generation got to proving z_RMP == z_MP, NOT itself a
# certified bound the way `gap` (UB - LB) is.
print(f"rmp_gap (z_RMP - LB, not certified): {dw_solution.rmp_gap:.3f}")
#print(f"compact-model exact objective (from 'Build and solve' above) = {solution.objective:.3f}")
print(f"whole_module_feasible: {dw_solution.whole_module_feasible}")
#dw_solution.per_vehicle
print(
    f"Total sojourn bracket: "
    f"[{dw_solution.total_sojourn_LB:.2f}, "
    f"{dw_solution.per_vehicle['sojourn_min'].sum():.2f}]"
)

[colgen] iter 1: z_RMP=1267.776, best_LB=882.491, gap=385.285, exact=True, candidates=40, columns_added=40, total_columns=118
[colgen] iter 2: z_RMP=1249.353, best_LB=899.485, gap=349.867, exact=True, candidates=40, columns_added=40, total_columns=158
[colgen] iter 3: z_RMP=1237.077, best_LB=936.097, gap=300.980, exact=True, candidates=40, columns_added=40, total_columns=198
[colgen] iter 4: z_RMP=1219.839, best_LB=973.189, gap=246.649, exact=True, candidates=40, columns_added=40, total_columns=238
[colgen] iter 5: z_RMP=1213.505, best_LB=1019.826, gap=193.679, exact=True, candidates=38, columns_added=38, total_columns=276
[colgen] iter 6: z_RMP=1210.052, best_LB=1063.068, gap=146.984, exact=True, candidates=36, columns_added=36, total_columns=312
[colgen] iter 7: z_RMP=1205.602, best_LB=1063.068, gap=142.534, exact=True, candidates=36, columns_added=36, total_columns=348
[colgen] iter 8: z_RMP=1202.145, best_LB=1111.632, gap=90.513, exact=True, candidates=40, columns_added=40, total_c

In [35]:
print(f"Sim total sojourn (finished): {sim_total_sojourn:.2f} min")
print(f"Sim mean sojourn (finished):  {sim_mean_sojourn:.2f} min")
dw_solution.mean_sojourn_LB

Sim total sojourn (finished): 541.80 min
Sim mean sojourn (finished):  41.68 min


30.442782458044313

In [29]:
print(f"Sim total sojourn (finished): {sim_total_sojourn:.2f} min")
print(f"Sim mean sojourn (finished):  {sim_mean_sojourn:.2f} min")
dw_solution.mean_sojourn_LB

Sim total sojourn (finished): 236.50 min
Sim mean sojourn (finished):  47.30 min


40.4628576492847

In [42]:

cl_model = build_cl_model(vehicles, cl_station, break_symmetry=True, bound_departures=True,
delta=DELTA, horizon_minutes=OFFLINE_HORIZON, tie_break=False)
solve_cl_model(cl_model, mip_gap=1e-3, time_limit=None, verbose=True, cutoff=None)
solution = extract_solution(cl_model)

print(f"status={solution.status}, gap={solution.mip_gap:.2%}, runtime={solution.runtime:.2f}s")
print(f"objective (sum D_j) = {solution.objective:.3f} slots, mean sojourn = {solution.mean_sojourn:.3f} min")
solution.per_vehicle

Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.001
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1145G7 @ 2.60GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0.001

Academic license 2616613 - for non-commercial use only - registered to m.___@mail.utoronto.ca
Optimize a model with 11335 rows, 3733 columns and 264444 nonzeros (Min)
Model fingerprint: 0x5b8a1b6f
Model has 1532 linear objective coefficients and an objective constant of 1560
Variable types: 2298 continuous, 1435 integer (1195 binary)
Coefficient statistics:
  Matrix range     [3e-02, 2e+02]
  Objective range  [1e+00, 6e+01]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 3e+02]

Found heuristic solution: objective 1560.0000000
Presolve removed 1634 rows and 525 columns
Presolve time: 1.00s
Presolved: 9701 rows

,vehicle_id,arrival,served,pile,connector,start_slot,departure_slot,sojourn_min,energy_kwh,energy_required_kwh
0,0,2.0,True,0.0,0.0,1.0,13.0,24.0,35.926494,35.926494
1,1,6.0,True,0.0,1.0,3.0,16.0,26.0,29.388572,29.388572
2,2,8.0,False,NaN,NaN,60.0,60.0,112.0,0.000000,38.487826
3,3,10.0,True,1.0,0.0,5.0,14.0,18.0,34.438342,34.438342
4,4,14.0,True,1.0,1.0,7.0,24.0,34.0,42.910430,42.910430
5,5,26.0,True,0.0,0.0,13.0,25.0,24.0,37.551944,37.551944
6,6,28.0,True,1.0,1.0,26.0,41.0,54.0,35.521619,35.521619
7,7,32.0,True,0.0,1.0,17.0,32.0,32.0,40.036083,40.036083
8,8,44.0,True,1.0,0.0,22.0,35.0,26.0,38.527586,38.527586
9,9,46.0,False,NaN,NaN,60.0,60.0,74.0,0.000000,47.784760


In [12]:

cl_model = build_cl_model(vehicles, cl_station, break_symmetry=True, bound_departures=True,
delta=DELTA, horizon_minutes=OFFLINE_HORIZON, tie_break=False)
solve_cl_model(cl_model, mip_gap=5e-3, time_limit=None, verbose=True, cutoff=None)
solution = extract_solution(cl_model)

print(f"status={solution.status}, gap={solution.mip_gap:.2%}, runtime={solution.runtime:.2f}s")
print(f"objective (sum D_j) = {solution.objective:.3f} slots, mean sojourn = {solution.mean_sojourn:.3f} min")
solution.per_vehicle

Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.005
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1145G7 @ 2.60GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  0.005

Academic license 2616613 - for non-commercial use only - registered to m.___@mail.utoronto.ca
Optimize a model with 5520 rows, 2572 columns and 40040 nonzeros (Min)
Model fingerprint: 0x5de6fd04
Model has 1144 linear objective coefficients and an objective constant of 960
Variable types: 1716 continuous, 856 integer (616 binary)
Coefficient statistics:
  Matrix range     [2e-02, 2e+02]
  Objective range  [1e+00, 1e+02]
  Bounds range     [1e+00, 2e+02]
  RHS range        [1e+00, 4e+02]

Found heuristic solution: objective 960.0000000
Presolve removed 1131 rows and 559 columns
Presolve time: 0.09s
Presolved: 4389 rows, 2013

,vehicle_id,arrival,served,pile,connector,start_slot,departure_slot,sojourn_min,energy_kwh,energy_required_kwh
0,0,3.438384,True,0.0,0.0,4.0,29.0,25.561616,39.681788,39.681788
1,1,17.464539,True,0.0,1.0,18.0,55.0,37.535461,51.053306,51.053306
2,2,24.296597,True,0.0,0.0,33.0,57.0,32.703403,28.869372,28.869372
3,3,31.411370,False,NaN,NaN,120.0,120.0,88.588630,0.000000,53.854608
4,4,44.757326,True,0.0,1.0,55.0,76.0,31.242674,34.778222,34.778222
5,5,78.553507,True,0.0,1.0,79.0,98.0,19.446493,34.124377,34.124377
6,6,86.074832,True,0.0,0.0,87.0,117.0,30.925167,38.012206,38.012206
7,7,97.833511,False,NaN,NaN,120.0,120.0,22.166489,0.000000,35.817405


In [11]:
solution.total_sojourn

219.33986859282697

In [ ]:
from offline_cl_opt import solve_cl_model_adaptive

result = solve_cl_model_adaptive(vehicles, cl_station, delta=DELTA, break_symmetry=True,
                        progress=True, verbose=False,
                        bound_departures=True, cutoff=None,
                        time_limit=None,
                        horizon_minutes=OFFLINE_HORIZON, mip_gap=5e-3,
                        warm_start_from_conservative=False)
adaptive_solution = extract_solution(result.cl_model)

print(f"converged={result.converged}, iterations={result.iterations}")
print(f"objective history: {result.objective_history}")
print(f"pile-slots promoted to integer: {len(result.integer_pile_slots)}")
print(f"adaptive objective = {adaptive_solution.objective:.3f} slots, mean sojourn = {adaptive_solution.mean_sojourn:.3f} min")
adaptive_solution.per_vehicle

In [ ]:
from offline_opt import (
    StationSpec,
    build_offline_model,
    build_relaxed_model,
    default_horizon_minutes,
    extract_solution,
    solve_offline_model,
    taper_time_constant,
    vehicles_from_evs,
)
offline_station = StationSpec(
    n_piles=SCENARIO["n_piles"],
    n_connectors=SCENARIO["n_connectors"],
    n_modules=SCENARIO["n_modules"],
    p_module=SCENARIO["p_module"],
)
# Same measured-phase-only, arrival-time-shifted vehicle set as the
# offline_cl_opt cell above (offline_opt.VehicleData is a different, also
# frozen, class with its own `a` field -- not interchangeable with
# offline_cl_opt.VehicleData, so this re-derives from the simulator EVs
# rather than reusing that cell's `vehicles` list).
vehicles = [
    dataclasses.replace(v, a=v.a - env.engine.warmup_period)
    for v in vehicles_from_evs(env.engine.metrics.arrived_post_warmup)
]
tau = taper_time_constant()
ip_model = build_offline_model(
    vehicles,
    offline_station,
    delta=DELTA,
    horizon_minutes=OFFLINE_HORIZON,
    tau=tau,
    break_pile_symmetry=True,
    tie_break=False,
)
solve_offline_model(ip_model, mip_gap=MIP_GAP, time_limit=None, verbose=True)
ip_sol = extract_solution(ip_model)
print("Done solving IP")